# Project Evals — Continue / Schedule / End

Use the **OpenAI Evals API** to test how well a prompted model classifies the next recruiter action on our labeled `sms_conversations.json` data.

This mirrors **Part A** of the class notebook, but the task is our project's task (not e-commerce complaints).

At the end there is also a **bonus section** that evaluates our actual multi-agent system (`get_main_agent_response`) locally.

## Setup

### Install dependencies (run once)
```powershell
pip install openai python-dotenv scikit-learn pandas matplotlib seaborn
```

In [26]:
# TODO: Import OpenAI (from openai), load_dotenv (from dotenv), os, json.
from openai import OpenAI
from dotenv import load_dotenv
import os

In [27]:
# TODO: Call load_dotenv() and read OPENAI_API_KEY from os.getenv.
#       Print only the first few characters to verify (never the full key).

load_dotenv()  # Loads variables from .env

openai_key = os.getenv("OPENAI_API_KEY")
print(openai_key[:5])  # Just to check, don't print the full key!

sk-pr


In [28]:
# TODO: Initialize the OpenAI client: client = OpenAI().
client = OpenAI()

## Prepare the test data

Convert `Relevant Files/sms_conversations.json` into a **JSONL** file where each line is one test item. We classify the next recruiter action given the conversation so far.

Each line will look like:
```json
{ "item": { "conversation_history": "recruiter: ...\ncandidate: ...", "correct_label": "continue" } }
```

In [29]:
# TODO: Open '../Relevant Files/sms_conversations.json' and json.load() it into a list of conversations.
# TODO: Open '../Relevant Files/sms_conversations.json' and json.load() it into a list of conversations.
import json

# Open and load the conversations JSON file:
with open("../Relevant Files/sms_conversations.json", encoding="utf-8") as conversations_file:
    conversations = json.load(conversations_file)

print(json.dumps(conversations[0], indent=4))

{
    "conversation_id": 1,
    "candidate_phone": "+1-555-0201",
    "recruiter_phone": "+1-555-0000",
    "start_time_utc": "2024-04-03T15:12:00Z",
    "turns": [
        {
            "turn_id": 1,
            "speaker": "recruiter",
            "timestamp_utc": "2024-04-03T15:12:00Z",
            "text": "Thanks for applying to our Python Developer opening. What kinds of Python projects have you worked on recently?",
            "label": "continue"
        },
        {
            "turn_id": 2,
            "speaker": "candidate",
            "timestamp_utc": "2024-04-03T15:13:19Z",
            "text": "I've been using Python professionally for five years, mostly for data analysis.",
            "label": null
        },
        {
            "turn_id": 3,
            "speaker": "recruiter",
            "timestamp_utc": "2024-04-03T15:15:01Z",
            "text": "Our engineering manager can interview you Wednesday at 10\u202fAM or Thursday at 2\u202fPM. Which works best?",
         

In [30]:
# TODO: Helper function `format_history(turns) -> str`.
#
# INPUT — `turns`: a slice of a conversation's `turns` list (from sms_conversations.json).
#   Each turn is a dict with at least these keys:
#       { 'speaker': 'recruiter' | 'candidate', 'text': '...', 'label': ... }
#   (we only use `speaker` and `text` here — ignore the rest)
#
# WHERE IT COMES FROM — the NEXT cell. For each labeled recruiter turn we want to predict,
#   we take all turns BEFORE it (the conversation so far) and pass that slice in here.
#
# OUTPUT — a single string, one line per turn, like:
#       recruiter: Thanks for applying...
#       candidate: I've been using Python...
#       recruiter: Could you tell me more about your projects?
#
# Implementation hint: loop over turns and join f"{t['speaker']}: {t['text']}" with newlines.

def format_history(turns):
    return "\n".join(
        f"{turn['speaker']}: {turn['text']}"
        for turn in turns
        )

In [31]:
print(format_history(conversations[0]["turns"]))

recruiter: Thanks for applying to our Python Developer opening. What kinds of Python projects have you worked on recently?
candidate: I've been using Python professionally for five years, mostly for data analysis.
recruiter: Our engineering manager can interview you Wednesday at 10 AM or Thursday at 2 PM. Which works best?
candidate: I can't at that time—I'm busy.
recruiter: No problem. How about Thursday at 4 PM instead?
candidate: Monday at 3 PM is good.
recruiter: Great, your interview is confirmed. You'll receive a calendar invite shortly.


In [32]:
# TODO: Build `test_items` — the list of test cases we'll evaluate on.
#
# A "test case" = (the conversation so far) + (the correct label for what the recruiter does next).
#
# For each conversation in the data:
#   walk through `conversation['turns']` in order, tracking the index i.
#   Whenever turn i is a RECRUITER turn AND its `label` is not None:
#       - prior_turns = conversation['turns'][:i]   # everything BEFORE this turn
#       - history    = format_history(prior_turns)   # the helper from the cell above
#       - label      = turn['label']
#       - append { 'conversation_history': history, 'correct_label': label } to test_items
#
# Note: candidate turns have label=None — skip them.
# Note: the very first recruiter turn has prior_turns=[] → history is just an empty string. That's fine.
#
# (We'll reuse `test_items` later in the bonus section too.)

test_items = []

for conversation in conversations:
    turns = conversation["turns"]

    for turn_index, turn in enumerate(turns):
        speaker = turn["speaker"]
        label = turn["label"] # Not null if recruiter

        if speaker == "recruiter" and label is not None:
            prior_turns = turns[:turn_index] # Everything BEFORE this turn
            history = format_history(prior_turns)
            test_items.append({'conversation_history': history, 'correct_label': label}) 



In [33]:
# TODO: Write `test_items` to 'sms_conversations_test.jsonl' — one JSON object per line,
#       wrapped as { 'item': {...} } (the format OpenAI Evals expects).

with open("sms_conversations_test.jsonl", "w" , encoding="utf-8") as test_file:
    for item in test_items:
        json.dump({"item": item}, test_file)
        test_file.write("\n")

## Define the task

The instructions tell the model what to output. We want a single word: `continue`, `schedule`, or `end`.

In [34]:
# TODO: Define `instructions` — a system prompt that tells the model to classify the next
#       recruiter action as exactly one of 'continue', 'schedule', or 'end' and respond with only that word.
instructions = """
You are a classifier that labels the next recruiter action in an SMS recruitment
chatbot for a Python Developer position.

Given a conversation history between a recruiter and a candidate, decide what the
recruiter should do NEXT, and output exactly one of three labels:

    continue  — Keep the conversation going. Use when the candidate is still sharing
                background, asking questions about the role, or the conversation is
                in its early stages and there is not yet enough context to schedule
                an interview. Also use when the candidate cannot make a proposed
                time slot and an alternative is needed but none has been offered yet.

    schedule  — The recruiter should propose, validate, or confirm an interview slot.
                Use when the candidate has shown interest and shared enough background,
                the candidate brings up availability or specific dates, the recruiter
                is offering time slots, or an interview is being confirmed.

    end       — The conversation should be closed. Use when the candidate explicitly
                declines (not interested, already found a job, asks to stop), or when
                an interview has just been confirmed and there is nothing left to
                discuss.

OUTPUT RULES (strict — your output is graded by exact string match):
 - Reply with EXACTLY one word: continue, schedule, or end.
 - Lowercase only.
 - No punctuation, no quotes, no explanation, no whitespace before or after.

EXAMPLES:

  History:
  recruiter: Thanks for applying to our Python Developer opening. Could you tell me about your recent Python work?
  candidate: I've been using Python professionally for five years, mostly for data analysis and building internal
  tools.
  Answer: continue

  History:
  recruiter: Thanks for sharing your background — sounds like a great fit.
  candidate: Thanks! When would the interview be?
  Answer: schedule

  History:
  recruiter: Would you be available for an interview next week?
  candidate: Actually, I just accepted another offer last week, so I have to withdraw. Thanks anyway.
  Answer: end

Now classify the following conversation. Remember: reply with only one word.
"""

In [35]:
test_items[2]

{'conversation_history': "recruiter: Thanks for applying to our Python Developer opening. What kinds of Python projects have you worked on recently?\ncandidate: I've been using Python professionally for five years, mostly for data analysis.\nrecruiter: Our engineering manager can interview you Wednesday at 10\u202fAM or Thursday at 2\u202fPM. Which works best?\ncandidate: I can't at that time—I'm busy.",
 'correct_label': 'schedule'}

In [38]:
# TODO: Quick sanity check — call client.chat.completions.create with one sample conversation_history.
#       Print the model's reply to verify it's one of the three labels.

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": test_items[2]["conversation_history"]}
    ]
)

print(completion.choices[0].message.content)

continue


## Create the eval

An eval = a **data schema** (`data_source_config`) + **graders** (`testing_criteria`). We use a `string_check` grader with `eq` to compare the model output to `correct_label`.

In [39]:
# TODO: Call client.evals.create(...).
#       - name: something like 'Recruiter Action Routing'
#       - data_source_config: type='custom', item_schema with properties conversation_history (string) and correct_label (string), include_sample_schema=True
#       - testing_criteria: a string_check grader comparing {{ sample.output_text }} eq {{ item.correct_label }}
#       Save eval_obj.id.

eval_obj = client.evals.create(
    name = "Recruiter Action Routing",

    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "conversation_history": {"type": "string"},
                "correct_label": {"type": "string"},
            },
            "required": ["conversation_history", "correct_label"],
        },
        "include_sample_schema": True
    },
    testing_criteria=[
        {
            "type": "string_check",
            "name": "Match output to human label",
            "input": "{{ sample.output_text }}",
            "operation": "eq",
            "reference": "{{ item.correct_label }}"
        }
    ],
)

print(eval_obj)
print(eval_obj.id)

EvalCreateResponse(id='eval_6a0705a24094819185e892dbbe702d33', created_at=1778845090, data_source_config=EvalCustomDataSourceConfig(schema_={'type': 'object', 'properties': {'item': {'type': 'object', 'properties': {'conversation_history': {'type': 'string'}, 'correct_label': {'type': 'string'}}, 'required': ['conversation_history', 'correct_label']}, 'sample': {'type': 'object', 'properties': {'model': {'type': 'string'}, 'choices': {'type': 'array', 'items': {'type': 'object', 'properties': {'message': {'type': 'object', 'properties': {'role': {'type': 'string', 'enum': ['assistant']}, 'content': {'type': ['string', 'array', 'null']}, 'refusal': {'type': ['boolean', 'null']}, 'tool_calls': {'type': ['array', 'null'], 'items': {'type': 'object', 'properties': {'type': {'type': 'string', 'enum': ['function']}, 'function': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'arguments': {'type': 'string'}}, 'required': ['name', 'arguments']}, 'id': {'type': 'string'}}, 'requir

## Upload the test JSONL

In [40]:
# TODO: client.files.create(file=open('sms_conversations_test.jsonl', 'rb'), purpose='evals').
#       Save file.id.

file = client.files.create(
    file=open("sms_conversations_test.jsonl", "rb"),
    purpose="evals"
)

print(file)
print(file.id)

FileObject(id='file-M3T4efymt5tEG6KHiWtrQU', bytes=20195, created_at=1778845219, filename='sms_conversations_test.jsonl', object='file', purpose='evals', status='processed', expires_at=None, status_details=None)
file-M3T4efymt5tEG6KHiWtrQU


## Create the eval run

In [41]:
# TODO: client.evals.runs.create(eval_obj.id, ...).
#       data_source = type='completions', model='gpt-4.1',
#         input_messages = template with system=instructions and user='{{ item.conversation_history }}',
#         source = file_id of the JSONL we uploaded.
#       Save run.id.

run = client.evals.runs.create(
    eval_obj.id, 

    name= "Recruiter text run",

    data_source={
        "type": "completions",
        "model": "gpt-4.1",
        "input_messages": {
            "type": "template",
            "template": [
                {"role": "developer", "content": instructions},
                {"role": "user", "content": "{{ item.conversation_history }}"}
            ],
        },
        "source": {"type": "file_id", "id": file.id},
    },
)

print(run)
print(run.id)

RunCreateResponse(id='evalrun_6a0707700de48191a36f0939755b9ca8', created_at=1778845552, data_source=CreateEvalCompletionsRunDataSource(source=SourceFileID(id='file-M3T4efymt5tEG6KHiWtrQU', type='file_id'), type='completions', input_messages=InputMessagesTemplate(template=[EasyInputMessage(content="\nYou are a classifier that labels the next recruiter action in an SMS recruitment\nchatbot for a Python Developer position.\n\nGiven a conversation history between a recruiter and a candidate, decide what the\nrecruiter should do NEXT, and output exactly one of three labels:\n\n    continue  — Keep the conversation going. Use when the candidate is still sharing\n                background, asking questions about the role, or the conversation is\n                in its early stages and there is not yet enough context to schedule\n                an interview. Also use when the candidate cannot make a proposed\n                time slot and an alternative is needed but none has been offered ye

## Analyze the results

In [42]:
# TODO: Retrieve the run with client.evals.runs.retrieve(eval_id=..., run_id=...).
#       Print run.status — wait until it becomes 'completed' before continuing.
run_retrieve = client.evals.runs.retrieve(
    eval_id=eval_obj.id, # YOUR_EVAL_ID
    run_id=run.id # YOUR_RUN_ID
    )


print(run_retrieve)
print(run_retrieve.status)

RunRetrieveResponse(id='evalrun_6a0707700de48191a36f0939755b9ca8', created_at=1778845552, data_source=CreateEvalCompletionsRunDataSource(source=SourceFileID(id='file-M3T4efymt5tEG6KHiWtrQU', type='file_id'), type='completions', input_messages=InputMessagesTemplate(template=[EasyInputMessage(content="\nYou are a classifier that labels the next recruiter action in an SMS recruitment\nchatbot for a Python Developer position.\n\nGiven a conversation history between a recruiter and a candidate, decide what the\nrecruiter should do NEXT, and output exactly one of three labels:\n\n    continue  — Keep the conversation going. Use when the candidate is still sharing\n                background, asking questions about the role, or the conversation is\n                in its early stages and there is not yet enough context to schedule\n                an interview. Also use when the candidate cannot make a proposed\n                time slot and an alternative is needed but none has been offered 

In [44]:
# TODO: Print passed / total from run.result_counts and the report_url.
print(f"Errored = {run_retrieve.result_counts.errored}")
print(f"Total = {run_retrieve.result_counts.total}")
print(f"Passed = {run_retrieve.result_counts.passed}")
print(f"Failed = {run_retrieve.result_counts.failed}")

Errored = 0
Total = 59
Passed = 33
Failed = 26


In [47]:
# TODO: List the output items: client.evals.runs.output_items.list(eval_id=..., run_id=...).
#       Extract y_true from item.datasource_item['correct_label']
#       and    y_pred from item.sample.output[0].content.strip().

items = client.evals.runs.output_items.list(
    eval_id=eval_obj.id,
    run_id=run.id
)

y_true = [item.datasource_item['correct_label'] for item in items]
y_pred = [item.sample.output[0].content.strip() for item in items]

print(y_true)
print(y_pred)

['end', 'continue', 'end', 'continue', 'schedule', 'schedule', 'end', 'schedule', 'continue', 'schedule', 'end', 'continue', 'continue', 'continue', 'end', 'end', 'schedule', 'continue', 'schedule', 'schedule', 'schedule', 'continue', 'schedule', 'continue', 'schedule', 'schedule', 'continue', 'end', 'schedule', 'end', 'continue', 'continue', 'continue', 'continue', 'end', 'end', 'end', 'continue', 'continue', 'continue', 'continue', 'end', 'continue', 'schedule', 'schedule', 'continue', 'schedule', 'end', 'continue', 'end', 'continue', 'continue', 'schedule', 'schedule', 'continue', 'continue', 'schedule', 'schedule', 'end']
['end', 'continue', 'end', 'continue', 'continue', 'continue', 'schedule', 'continue', 'continue', 'continue', 'schedule', 'continue', 'continue', 'continue', 'schedule', 'end', 'continue', 'continue', 'continue', 'continue', 'continue', 'continue', 'continue', 'continue', 'continue', 'continue', 'continue', 'schedule', 'continue', 'end', 'continue', 'continue', '

In [48]:
# TODO: from sklearn.metrics import accuracy_score — print accuracy_score(y_true, y_pred).
from sklearn.metrics import accuracy_score

print(f"accuracy_score: {accuracy_score(y_true, y_pred)}")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# TODO: from sklearn.metrics import confusion_matrix — compute it with labels=['continue','schedule','end'].
#       Wrap in a pandas DataFrame and print it.

In [ ]:
# TODO: Plot the confusion matrix with seaborn.heatmap (annot=True, fmt='d', cmap='Blues').
#       Label axes 'Predicted' / 'True' and give it a title.

---

## Bonus: Evaluate our actual multi-agent system

The cells above test a **single prompt** against `gpt-4.1` through OpenAI's cloud eval. That is useful as a baseline, but it doesn't test the code we actually wrote.

Below we run the **same test cases** through our real pipeline — `get_main_agent_response` — which routes through the Exit / Scheduling / Info advisors. Then we compute Accuracy + Confusion Matrix the same way.

In [ ]:
# TODO: Make our `app` package importable from this notebook.
#       import sys, pathlib
#       sys.path.append(str(pathlib.Path.cwd().parent))   # adds the project root to sys.path

In [ ]:
# TODO: from langchain_openai import ChatOpenAI
#       from app.modules.agents.main_agent import get_main_agent_response

In [ ]:
# TODO: Create the LLM our agent will use: llm = ChatOpenAI(model='gpt-4.1', temperature=0).

In [ ]:
# TODO: Loop over the same `test_items` we built earlier.
#       For each item:
#         - call get_main_agent_response(item['conversation_history'], llm)
#         - append the returned 'action' to y_pred_local
#         - append item['correct_label'] to y_true_local
#       (Tip: print progress every ~10 items — this is slower than the cloud eval.)

In [ ]:
# TODO: Print accuracy_score(y_true_local, y_pred_local).

In [ ]:
# TODO: confusion_matrix(y_true_local, y_pred_local, labels=['continue','schedule','end']) → DataFrame, then print.

In [ ]:
# TODO: seaborn.heatmap of the local confusion matrix. Title it 'Multi-Agent System — Confusion Matrix'.

## Compare

Side-by-side: how does our advisor-routed multi-agent system compare to a plain single-prompt call?

In [ ]:
# TODO: Print both accuracies together so we can compare:
#         Cloud eval (single prompt): ...
#         Multi-agent system        : ...